Imports:

In [3]:
import aerosandbox as asb
import aerosandbox.numpy as np

from wing_deflection import max_deflection

Constants, definitions that shouldn't change

In [4]:
wing_airfoil = asb.Airfoil("naca0001")
tail_airfoil = asb.Airfoil("naca0001")

E = 3e9
rho_balsa = 250 # kg/m3
rho = 1.225
nu = 1.5e-5
pi = np.pi
AR_v = 3

H_loc = 0.1 #distance of LE of horizontal stabilizer and vert stabilizer from the LE of the wing


Beginning the optimization environment, initializing some variables


In [5]:
opti = asb.Opti()

V = opti.variable(init_guess = 4, lower_bound = 0.1, upper_bound = 5)
AR = opti.variable(init_guess = 5, lower_bound = 2, upper_bound = 15)
S = opti.variable(init_guess = 0.05, lower_bound = 0.01, upper_bound = 0.5)
twist = opti.variable(init_guess = 2, lower_bound = 0, upper_bound = 10)
dihedral = opti.variable(init_guess = 5, lower_bound = 0, upper_bound=10)
AR_h = opti.variable(init_guess = 4, lower_bound = 2)
S_h = opti.variable(init_guess = 0.01, lower_bound = 0.00001)
S_v = opti.variable(init_guess = 0.005, lower_bound = 0.000001)
glide_angle = opti.variable(init_guess = 10, lower_bound=0)

Derived variables from the given optimized variables

In [6]:
b_total = np.sqrt(S * AR)
c = S / b_total
b = b_total / 2

b_h = np.sqrt(S_h * AR_h) / 2
c_h = S_h / (b_h * 2)

b_v = np.sqrt(S_v * AR_v)
c_v = S_v / b_v

Weight and structural model

In [24]:
thickness = 0.00025
boom_area = thickness * 0.01
margin = 0.001
I_formula = lambda chord, tau: chord * thickness ** 3 / 12

weight = (S + S_h + S_v) * thickness * rho_balsa + H_loc * rho_balsa * boom_area + margin

cumsum_weight = (c / 2 * S + (H_loc + c_h/2) * S_h + (H_loc + c_v/2) * S_v) * thickness * rho_balsa + H_loc / 2 * H_loc * boom_area * rho_balsa
COM = cumsum_weight / weight

# Structural Deflection Constraint (tau set to 0.05 dynamically)
N = 1.2
opti.subject_to(max_deflection(N * weight * 9.81, AR, S, E, I_formula = I_formula, tau=0.05) < 0.1 * b)
opti.subject_to(max_deflection(N * weight * 9.81, AR_h, S_h, E, I_formula = I_formula, tau=0.05) < 0.1 * b_h)
# opti.subject_to(max_deflection(N * weight * 9.81, AR_v, S_v, E, I_formula = I_formula, tau=0.05) < 0.02)


MX(fabs(opti1_lam_g_28))

Aerodynamic Constraints

In [25]:
import casadi as ca

#tail volume coefficients
hor_vol_coef = S_h * H_loc / (S * c)
ver_vol_coef = S_v * H_loc / (S * c)

opti.subject_to(hor_vol_coef > 0.3)
opti.subject_to(hor_vol_coef < 0.6)

opti.subject_to(ver_vol_coef > 0.02)
opti.subject_to(ver_vol_coef < 0.05)

#neutral point
a_w = 2 * pi / (1 + 2 / AR)
a_h = 2 * pi / (1 + 2 / AR_h)

np = c * (a_w / (4 * a_h) + hor_vol_coef * (1 + c / (4 * H_loc))) / (a_w / a_h + hor_vol_coef * c / H_loc )
opti.subject_to(np > COM)

#aerodynamic constraints

a0 = 2 * pi
e = 0.9
Re = V * c / nu

CL = a0 / (1 + a0 / (pi * AR * e)) * twist

CL_h = - (COM / c - 0.25) * CL / ((COM / H_loc - 1 - c / H_loc) * hor_vol_coef)
alpha_t = CL_h / (2 * pi)

CDi = CL**2 / (pi * AR * e) + CL_h ** 2 / (pi * AR_h * e)
CD0 = 1.3 * 2.656 / Re ** 0.5
CD = CDi + CD0

q = 1/2 * rho * V**2 * S
L = q * (CL + CL_h)
D = q * CD

opti.subject_to(weight * 9.81 * ca.cos(glide_angle * pi / 180) < L)
opti.subject_to(weight * 9.81 * ca.sin(glide_angle * pi / 180) > D)
opti.subject_to(CL < 1.4)

#spiral stability
spiral  = H_loc * dihedral / (b_total * CL)

opti.subject_to(spiral > 5)

MX(fabs(opti1_lam_g_37))

Objective function and solving

In [26]:
opti.minimize(glide_angle)

# --- Solve the Problem ---
sol = opti.solve(behavior_on_failure="return_last", max_iter=100,
    options={"ipopt.linear_solver": "mumps",
    "ipopt.mumps_mem_percent":300,
    })


This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:      111
Number of nonzeros in Lagrangian Hessian.............:       35

Total number of variables............................:        9
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:       37
        inequality constraints with only lower bounds:       15
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:       22

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  1.0000000e+01 7.30e+00 4.50e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

Output statements

In [32]:
print("\n" + "="*40)
print("       Grass'S GLIDER DESIGN METRICS     ")
print("="*40)
print(f"Optimization Status : {sol.stats()['return_status']}")
print(f"Total Glider Mass   : {sol.value(weight) * 1000:.2f} grams")
print(f"Design Velocity     : {sol.value(V):.2f} m/s")
print("-"*40)
print(f"Main Wing Area (S)  : {sol.value(S):.4f} m^2")
print(f"Aspect Ratio (AR)   : {sol.value(AR):.2f}")
print(f"Total Wingspan (b)  : {sol.value(b_total):.2f} m")
print(f"Wing Chord (c)      : {sol.value(c)*1000:.1f} mm")
print(f"Wing Dihedral       : {sol.value(dihedral):.1f} degrees")
print(f"Wing Twist          : {sol.value(twist):.1f} degrees")
print(f"Glide Angle          : {sol.value(glide_angle):.1f} degrees")
print("-"*40)
print(f"Horizontal Tail Area: {sol.value(S_h):.4f} m^2")
print(f"Vertical Tail Area  : {sol.value(S_v):.4f} m^2")

print(f"Horizontal Tail AR: {sol.value(AR_h):.4f} m^2")
print(f"Horizontal Tail b: {sol.value(b_h * 2):.4f} m")
print(f"Horizontal Tail c: {sol.value(c_h):.4f} m")

print(f"Vertical Tail AR: {sol.value(AR_v):.4f} m^2")
print(f"Vertical Tail b: {sol.value(b_v):.4f} m")
print(f"Vertical Tail c: {sol.value(c_v):.4f} m")

print(f"Tail Boom Length    : {sol.value(H_loc):.2f} m")
print("-"*40)
print(f"Aerodynamic Lift    : {sol.value(L):.3f} N")
print(f"Required Weight Lift: {sol.value(weight)*9.81:.3f} N")
print(f"Operating CL        : {sol.value(CL):.3f}")
print("-"*40)
print(f"Horiz. Vol. Coeff.  : {sol.value(hor_vol_coef):.3f}  (Target: 0.3 - 0.6)")
print(f"Vert. Vol. Coeff.   : {sol.value(ver_vol_coef):.3f}  (Target: 0.02 - 0.05)")
print(f"Spiral Criterion (B): {sol.value(spiral):.3f}")
print(sol.value(max_deflection(N * weight * 9.81, AR, S, E, I_formula = I_formula, tau=0.05)))

print(sol.value(COM))


       Grass'S GLIDER DESIGN METRICS     
Optimization Status : Solve_Succeeded
Total Glider Mass   : 2.15 grams
Design Velocity     : 1.68 m/s
----------------------------------------
Main Wing Area (S)  : 0.0145 m^2
Aspect Ratio (AR)   : 4.42
Total Wingspan (b)  : 0.25 m
Wing Chord (c)      : 57.3 mm
Wing Dihedral       : 9.3 degrees
Wing Twist          : 0.1 degrees
Glide Angle          : 5.2 degrees
----------------------------------------
Horizontal Tail Area: 0.0025 m^2
Vertical Tail Area  : 0.0004 m^2
Horizontal Tail AR: 7.9406 m^2
Horizontal Tail b: 0.1409 m
Horizontal Tail c: 0.0177 m
Vertical Tail AR: 3.0000 m^2
Vertical Tail b: 0.0353 m
Vertical Tail c: 0.0118 m
Tail Boom Length    : 0.10 m
----------------------------------------
Aerodynamic Lift    : 0.021 N
Required Weight Lift: 0.021 N
Operating CL        : 0.613
----------------------------------------
Horiz. Vol. Coeff.  : 0.300  (Target: 0.3 - 0.6)
Vert. Vol. Coeff.   : 0.050  (Target: 0.02 - 0.05)
Spiral Criterion (

In [28]:
airplane = asb.Airplane(
    name="Peter's Glider",
    xyz_ref=[0, 0, 0],  
    wings=[
        asb.Wing(
            name="Main Wing",
            symmetric=True,  
            xsecs=[  
                asb.WingXSec(  
                    xyz_le=[0, 0, 0],  
                    chord=c,
                    twist=twist,  
                    airfoil=wing_airfoil,  
                ),
                asb.WingXSec(  
                    xyz_le=[0, b, b * dihedral * pi / 180],
                    chord=c,
                    twist=twist,
                    airfoil=wing_airfoil,
                ),
            ],
        ),
        asb.Wing(
            name="Horizontal Stabilizer",
            symmetric=True,
            xsecs=[
                asb.WingXSec(
                    xyz_le=[0, 0, 0],
                    chord=c_h,
                    twist=0,
                    airfoil=tail_airfoil,
                ),
                asb.WingXSec(
                    xyz_le=[0, b_h, 0], chord=c_h, twist=0, airfoil=tail_airfoil
                ),
            ],
        ).translate([H_loc, 0, 0]),
        asb.Wing(
            name="Vertical Stabilizer",
            symmetric=False,
            xsecs=[
                asb.WingXSec(
                    xyz_le=[0, 0, 0],
                    chord=c_v,
                    twist=0,
                    airfoil=tail_airfoil,
                ),
                asb.WingXSec(
                    xyz_le=[0, 0, b_v], chord=c_v, twist=0, airfoil=tail_airfoil
                ),
            ],
        ).translate([H_loc, 0, 0]),
    ],
)

optimized_airplane = sol.value(airplane)

# This will open up an interactive browser window showing your full 3D layout!
optimized_airplane.draw()

Widget(value='<iframe src="http://localhost:37109/index.html?ui=P_0x7838ac59cb60_2&reconnect=auto" class="pyvi…

PolyData (0x7838bf692bc0)
  N Cells:    725
  N Points:   730
  N Strips:   0
  X Bounds:   -4.973e-25, 1.177e-01
  Y Bounds:   -1.267e-01, 1.267e-01
  Z Bounds:   -3.320e-04, 3.535e-02
  N Arrays:   0